# 岭回归：L2 正则化驯服共线性
> 难度标签：初级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：02_监督学习/回归 | 源文件：岭回归.py | 核心功能：L2 正则化、alpha 调参、RidgeCV 自动选参

## 概述

线性回归在多重共线性（特征间高度相关）下会"翻车"——系数估计变得极不稳定，某些系数可能飙到很大。岭回归通过在损失函数中加入 L2 惩罚项（α||w||²），把系数向 0 方向"收缩"，让模型更稳定。

数学上，岭回归的损失函数是：||y - Xw||² + α||w||²。α=0 退化为普通线性回归，α→∞ 所有系数趋于 0。

脚本在含共线性的人造数据上对比了线性回归和岭回归，并演示了 RidgeCV 自动选择最优 alpha。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, r2_score
# --- 导入库 ---
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 岭回归的目标函数

**代码对应**：`Ridge(alpha=1.0).fit(X_train, y_train)` 使用 L2 正则化。

岭回归在 OLS 基础上加入 L2 惩罚项：

$$J(\mathbf{w}) = \|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha\|\mathbf{w}\|_2^2$$

其中 $\alpha \geq 0$ 为正则化强度参数。

### 2. 闭式解推导

对 $J(\mathbf{w})$ 求梯度并令其为零：

$$\frac{\partial J}{\partial \mathbf{w}} = -2\mathbf{X}^T(\mathbf{y} - \mathbf{X}\mathbf{w}) + 2\alpha\mathbf{w} = \mathbf{0}$$

$$\mathbf{X}^T\mathbf{X}\mathbf{w} + \alpha\mathbf{w} = \mathbf{X}^T\mathbf{y}$$

$$(\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})\mathbf{w} = \mathbf{X}^T\mathbf{y}$$

岭回归解为：

$$\hat{\mathbf{w}}_{\text{ridge}} = (\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

**关键性质**：即使 $\mathbf{X}^T\mathbf{X}$ 不可逆（多重共线性或 $p > n$），$\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I}$ 总是可逆的（因为加了 $\alpha\mathbf{I}$ 使所有特征值至少为 $\alpha > 0$）。这就是"岭"（Ridge）名称的来源——在矩阵对角线上加了一道"山岭"。

### 3. 奇异值分解视角

对 $\mathbf{X}$ 做 SVD：$\mathbf{X} = \mathbf{U}\mathbf{D}\mathbf{V}^T$，其中 $\mathbf{D} = \text{diag}(d_1, \ldots, d_p)$。

OLS 解中第 $j$ 个分量为 $\hat{w}_j^{\text{OLS}} = \frac{\mathbf{u}_j^T\mathbf{y}}{d_j}$，当 $d_j \approx 0$（共线性）时值极大。

岭回归解中第 $j$ 个分量为：

$$\hat{w}_j^{\text{ridge}} = \frac{d_j}{d_j^2 + \alpha} \mathbf{u}_j^T\mathbf{y} = \frac{d_j^2}{d_j^2 + \alpha} \cdot \frac{\mathbf{u}_j^T\mathbf{y}}{d_j}$$

**收缩因子** $s_j = \frac{d_j^2}{d_j^2 + \alpha} \in [0, 1]$：当 $d_j$ 大时 $s_j \approx 1$（保留），当 $d_j$ 小时 $s_j \approx 0$（收缩）。岭回归自动对小奇异值方向进行更强的收缩。

### 4. 偏差-方差权衡

**代码对应**：代码中 alpha 从 0.001 到 1000 的对比实验展示了这一权衡。

$$\text{MSE}(\hat{\mathbf{w}}_{\text{ridge}}) = \underbrace{\alpha^2 \mathbf{w}^T(\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})^{-2}\mathbf{w}}_{\text{Bias}^2} + \underbrace{\sigma^2 \text{tr}(\mathbf{X}^T\mathbf{X}(\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})^{-2})}_{\text{Variance}}$$

- $\alpha = 0$：无偏差，高方差（OLS）
- $\alpha \to \infty$：高偏差（系数全为 0），低方差
- 最优 $\alpha^*$ 在两者之间，通过交叉验证选择

**代码对应**：`RidgeCV(alphas=alphas, cv=5)` 自动在多个 alpha 上做交叉验证，选择测试 R² 最高的 alpha。

### 5. 与贝叶斯推断的联系

岭回归等价于对权重施加**高斯先验**：

$$\mathbf{w} \sim \mathcal{N}(\mathbf{0}, \frac{1}{\alpha}\mathbf{I})$$

在先验 $P(\mathbf{w})$ 和似然 $P(\mathbf{y}|\mathbf{X}, \mathbf{w})$ 下，MAP（最大后验）估计为：

$$\hat{\mathbf{w}}_{\text{MAP}} = \arg\max_{\mathbf{w}} \ln P(\mathbf{w}|\mathbf{y}) = \arg\min_{\mathbf{w}} \left[\|\mathbf{y} - \mathbf{X}\mathbf{w}\|_2^2 + \alpha\|\mathbf{w}\|_2^2\right]$$

这正是岭回归的目标函数。$\alpha$ 越大，先验越"紧"（系数越接近 0）。

### 1. 生成含共线性的数据

运行 1. 生成含共线性的数据 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
n, p = 200, 10
X = np.random.randn(n, p)
# 让部分特征高度相关（模拟多重共线性）
X[:, 1] = X[:, 0] * 0.9 + np.random.randn(n) * 0.1
X[:, 2] = X[:, 0] * 0.8 + np.random.randn(n) * 0.1
y = 3 * X[:, 0] + 2 * X[:, 3] - 1.5 * X[:, 5] + np.random.randn(n) * 2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 2. 线性回归 vs 岭回归

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)

print("=== 线性回归 vs 岭回归 ===")
print(f"{'':>20} {'线性回归':>12} {'岭回归':>12}")
print(f"{'训练 R²':>20} {lr.score(X_train, y_train):>12.4f} {ridge.score(X_train, y_train):>12.4f}")
print(f"{'测试 R²':>20} {lr.score(X_test, y_test):>12.4f} {ridge.score(X_test, y_test):>12.4f}")

print(f"\n系数对比:")
print(f"{'特征':>8} {'线性回归':>12} {'岭回归':>12}")
for i in range(p):
    print(f"{'X'+str(i):>8} {lr.coef_[i]:>12.4f} {ridge.coef_[i]:>12.4f}")

### 3. alpha 参数选择

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== alpha 参数影响 ===")
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
for alpha in alphas:
    r = Ridge(alpha=alpha).fit(X_train, y_train)
    print(f"  alpha={alpha:>8}: 训练R²={r.score(X_train, y_train):.4f}, "
# --- 数值计算 ---
          f"测试R²={r.score(X_test, y_test):.4f}, 系数L2范数={np.linalg.norm(r.coef_):.4f}")

### 4. 交叉验证选 alpha

运行 4. 交叉验证选 alpha 的代码，观察算法在该环节的行为。

In [ ]:
# RidgeCV 自动在多个 alpha 上做交叉验证
ridge_cv = RidgeCV(alphas=alphas, scoring="r2", cv=5)
ridge_cv.fit(X_train, y_train)
print(f"\n=== 交叉验证最优 alpha ===")
print(f"最佳 alpha: {ridge_cv.alpha_}")
print(f"测试 R²: {ridge_cv.score(X_test, y_test):.4f}")

### 5. 正则化原理

运行 5. 正则化原理 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 岭回归原理 ===")
print("损失函数: ||y - Xw||² + α||w||²")
print("α=0 退化为普通线性回归；α→∞ 所有系数趋于 0")
print("L2 正则化将系数收缩向 0，但不会精确等于 0（不做特征选择）")

print("\n=== 岭回归要点 ===")
print("- 适用于：多重共线性、特征数 > 样本数、系数过大")
print("- alpha 越大正则化越强，模型越简单（偏差大、方差小）")
print("- 不能做特征选择（系数不会精确为 0）")
print("- 特征缩放很重要（正则化对系数大小敏感）")

## 关键代码解释

### 系数收缩效果

```python
lr = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
```

对比两个模型的系数可以清楚看到：岭回归的系数绝对值明显更小，且在共线性特征之间更"均匀"。L2 正则化不是把系数压到 0，而是让所有系数都缩小。

### RidgeCV 自动选参

```python
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0], scoring="r2", cv=5)
ridge_cv.fit(X_train, y_train)
```

RidgeCV 在多个 alpha 值上做留一法交叉验证（对 Ridge 有高效公式，无需逐折训练），自动选出最优 alpha。

## 使用示例

```python
from sklearn.linear_model import RidgeCV
ridge_cv = RidgeCV(alphas=[0.1, 1.0, 10.0], cv=5)
ridge_cv.fit(X_train, y_train)
print(f"最佳 alpha: {ridge_cv.alpha_}")
```

## 注意事项

1. **必须特征缩放**：正则化对系数大小敏感，不缩放会导致尺度大的特征惩罚更重
2. **不做特征选择**：L2 不会把系数精确压缩为 0
3. **alpha 搜索范围**：通常在 [0.001, 1000] 的对数网格上搜索
4. **解析解**：岭回归有闭合解 w = (X'X + αI)⁻¹X'y，计算很快

## 总结与延伸

以上代码展示了 **岭回归** 的完整流程。

**解读要点**：
- 关注 **R² 分数** 和 **MSE/MAE** 等回归指标
- R² 越接近 1 说明拟合越好
- 观察预测值 vs 真实值的散点图是否沿对角线分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **岭回归的贝叶斯解释**：L2 正则化等价于给权重施加高斯先验 w ~ N(0, α⁻¹I)
- **核岭回归**：结合核技巧处理非线性关系
- **Ridge vs SGDRegressor**：大数据集用 SGDRegressor(penalty="l2") 更高效
